# IWI experiment · Check the downloaded dataset before training

Reads every file written by `01_Download_Landsat_NL_MODIS` once, checks it, and writes three files next to `Dataset/`:

| file | content |
|---|---|
| `qc_report.csv` | one row per cluster: status of each file, share of no-data pixels, value ranges, issues and warnings |
| `train_clusters.csv` | the rows of the DHS table that pass every check, with the folder path: train on this list |
| `band_stats.csv` | mean, std, min and max of every band over those clusters (NL separately for DMSP and VIIRS), for normalisation |

**What is checked**

* **Structure.** The file opens; it has 224 × 224 pixels, the expected bands and band names, float32, EPSG:4326, north up and about 30 m pixels; it is centred on the cluster and its tags name this cluster; it has no NaN or inf; the three files share one grid. A file failing here is *broken*. Section 5 moves broken files aside, and running `01` again downloads them anew.
* **Content.** Share of no-data pixels (-999999), empty MODIS bands, values in range (NDVI −0.2…1, DMSP 0…63), constant Landsat bands, and a missing or out-of-range `iwi` label. These decide whether a cluster goes into `train_clusters.csv`, using the limits in Settings.
* **Warnings** (listed, not excluded). Many bright Landsat pixels (possible cloud), Landsat surface temperature outside 250–340 K, VIIRS values below −1 or above 1000, an NL sensor that does not match the year, identical Landsat patches in two clusters.

Nothing is deleted. Section 5 moves files only when you set `MOVE_BROKEN = True`.

Run it on a machine with many CPU cores (16 cores take a few minutes to read everything).

# Import libraries

In [ ]:
import os
import glob
import time
import shutil
import hashlib
import multiprocessing

# Same as in 01: the GPU image's PROJ_LIB has no proj.db, which breaks CRS handling in rasterio.
for _var in ('PROJ_LIB', 'PROJ_DATA'):
    if os.environ.get(_var) and not os.path.exists(os.path.join(os.environ[_var], 'proj.db')):
        os.environ.pop(_var)

import numpy as np
import pandas as pd
import rasterio
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# Settings

In [ ]:
CSV_PATH  = '/path/to/data/dhs_data_cluster.csv' # same as in 01
OUT_DIR   = '/path/to/data/Dataset/'
LOG_CSV   = '/path/to/data/download_log.csv'
QC_DIR    = '/path/to/data/'   # reports and _quarantine/ go here
MIN_YEAR  = 2000
LABEL     = 'iwi'

FILES = {'landsat': 'Landsat.tif', 'modis': 'MODIS_NDVI.tif', 'nl': 'NL.tif'}
BANDS = {'landsat': ['BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2', 'TEMP1'],
         'modis':   ['NDVI_%02d' % k for k in range(1, 24)],
         'nl':      ['NL']}
PATCH_PX = 224
NODATA   = -999999.0

# ---- what a cluster needs to go into train_clusters.csv (change and re-run section 3; no need to re-read)
MAX_NODATA = {'landsat': 0.0, 'modis': 0.0, 'nl': 0.0}   # largest share of pixels with no data in any band
                                                          # (MODIS: not counting bands that are empty)
MODIS_MAX_EMPTY_BANDS = 0      # 3 keeps the year-2000 clusters, whose NDVI_01-03 do not exist

# ---- warnings only
WARN_BRIGHT = 0.2              # share of Landsat pixels with blue reflectance > 0.3 (possible cloud)

N_WORKERS = len(os.sched_getaffinity(0))   # the cores this job was given
print('workers:', N_WORKERS)

# 1. Clusters that should have been downloaded

In [ ]:
TableDHS = pd.read_csv(CSV_PATH)
TableDHS = TableDHS.rename(columns={'ClusterID': 'cluster_id', 'year.of.interview': 'year',
                                    'month.of.interview': 'month', 'iwi_avg': 'iwi'})
TableDHS = TableDHS[TableDHS['year'] >= MIN_YEAR]
no_xy = TableDHS[['lon', 'lat']].isna().any(axis=1) | ((TableDHS['lon'] == 0) & (TableDHS['lat'] == 0))
TableDHS = TableDHS[~no_xy]
TableDHS = TableDHS[~TableDHS['cluster_id'].duplicated(keep='first')].reset_index(drop=True)
TableDHS['year'] = TableDHS['year'].astype(int)
print('clusters expected:', len(TableDHS))

def cluster_dir(cluster_id):
    return os.path.join(OUT_DIR, str(cluster_id).replace('/', '_'))

expected = {os.path.basename(cluster_dir(c)) for c in TableDHS['cluster_id']}
present = {e.name for e in os.scandir(OUT_DIR) if e.is_dir()}
unexpected_folders = sorted(present - expected)
print('folders in Dataset/:', len(present), '| expected but missing:', len(expected - present),
      '| not in the CSV:', len(unexpected_folders), unexpected_folders[:5])

# 2. Read and check every file

In [ ]:
def band_stats(a, nod):
    """count, sum, sum of squares, min, max of the valid pixels of each band (for band_stats.csv)."""
    out = np.zeros((a.shape[0], 5))
    for b in range(a.shape[0]):
        v = a[b][~nod[b]].astype(np.float64)
        out[b] = (v.size, v.sum(), (v * v).sum(), v.min(), v.max()) if v.size else (0, 0, 0, np.nan, np.nan)
    return out


def check_file(path, product, cid, year, lon, lat):
    """Check one GeoTIFF. Returns status ('ok' / 'broken'), report columns, issues, warnings, transform, band stats."""
    with rasterio.open(path) as src:
        a = src.read()
        crs, tr, names, tags, dtype = src.crs, src.transform, list(src.descriptions), src.tags(), src.dtypes[0]
    info, issues, warns = {}, [], []

    # ---- structure: a failure here means the file has to be downloaded again
    if a.shape != (len(BANDS[product]), PATCH_PX, PATCH_PX):
        issues.append('shape')
    if dtype != 'float32':
        issues.append('dtype')
    if crs is None or crs.to_epsg() != 4326:
        issues.append('crs')
    if not (tr.e < 0 and tr.b == 0 and tr.d == 0):
        issues.append('not north-up')
    if names != BANDS[product]:
        issues.append('band names')
    if tags.get('cluster_id') != cid or tags.get('year') != str(year):
        issues.append('tags of another cluster')
    if not np.isfinite(a).all():
        issues.append('NaN/inf')
    if not issues:
        col, row = ~tr * (lon, lat)
        if abs(col - PATCH_PX / 2) > 1.5 or abs(row - PATCH_PX / 2) > 1.5:
            issues.append('not centred on the cluster')
        px_x, px_y = tr.a * 111320 * np.cos(np.radians(lat)), -tr.e * 110574
        if not (25 < px_x < 35 and 25 < px_y < 35):
            issues.append('pixel size')
    if issues:
        return 'broken', info, [f'{product}: {i}' for i in issues], warns, tr, None

    # ---- content
    nod = a == NODATA
    if product == 'landsat':
        valid = ~nod.any(axis=0)
        info['landsat_nodata'] = 1 - float(valid.mean())
        info['landsat_levels'] = tags.get('levels', '')
        info['landsat_md5'] = hashlib.md5(a.tobytes()).hexdigest()
        if any(np.ptp(a[b][valid]) == 0 for b in range(a.shape[0]) if valid.any()):
            issues.append('constant band')
        if valid.any():
            info['landsat_bright'] = float(((a[0][valid] * 0.0000275 - 0.2) > 0.3).mean())
            info['landsat_temp_K'] = float(np.median(a[6][valid] * 0.00341802 + 149))
            if info['landsat_bright'] > WARN_BRIGHT:
                warns.append('bright pixels (cloud?)')
            if not 250 <= info['landsat_temp_K'] <= 340:
                warns.append('TEMP1 outside 250-340 K')
    elif product == 'modis':
        empty = nod.all(axis=(1, 2))
        info['modis_empty'] = int(empty.sum())
        info['modis_empty_bands'] = ' '.join(str(k + 1) for k in np.flatnonzero(empty))
        info['modis_nodata'] = float(nod[~empty].any(axis=0).mean()) if (~empty).any() else 1.0
        v = a[~nod]
        if v.size:
            info['modis_min'], info['modis_max'] = float(v.min()), float(v.max())
            if v.min() < -0.2001 or v.max() > 1.0001:
                issues.append('NDVI out of range')
        if tags.get('n_composites') and 23 - int(tags['n_composites']) != empty.sum():
            warns.append('empty bands differ from n_composites')
    else:
        sensor = tags.get('sensor', '')
        info['nl_sensor'] = sensor
        info['nl_nodata'] = float(nod.any(axis=0).mean())
        v = a[0][~nod[0]]
        if v.size:
            info['nl_min'], info['nl_max'] = float(v.min()), float(v.max())
            if sensor == 'DMSP' and (v.min() < 0 or v.max() > 63):
                issues.append('DMSP out of range')
            if sensor == 'VIIRS' and (v.min() < -1 or v.max() > 1000):
                warns.append('VIIRS below -1 or above 1000')
        if (year <= 2013) != (sensor == 'DMSP'):
            warns.append('sensor does not match year')
    return 'ok', info, [f'{product}: {i}' for i in issues], [f'{product}: {w}' for w in warns], tr, band_stats(a, nod)


def check_cluster(row):
    """Check the folder of one cluster. Returns (report row, band stats of the readable files)."""
    cid, year, lon, lat = row['cluster_id'], int(row['year']), float(row['lon']), float(row['lat'])
    folder = cluster_dir(cid)
    rec, issues, warns, grids, stats = {'cluster_id': cid, 'year': year}, [], [], {}, {}
    try:
        names = set(os.listdir(folder))
    except FileNotFoundError:
        names = None
        issues.append('folder missing')
    rec['part_files'] = sum(n.endswith('.part') for n in names) if names else 0
    rec['MB'] = 0.0
    for product, fname in FILES.items():
        path = os.path.join(folder, fname)
        if names is None or fname not in names:
            rec[f'{product}_status'] = 'missing'
            continue
        rec['MB'] += os.path.getsize(path) / 1e6
        try:
            status, info, iss, wrn, tr, st = check_file(path, product, cid, year, lon, lat)
        except Exception as e:
            status, info, iss, wrn, tr, st = 'unreadable', {}, [f'{product}: unreadable ({type(e).__name__})'], [], None, None
        rec[f'{product}_status'] = status
        rec.update(info)
        issues += iss
        warns += wrn
        if status == 'ok':
            grids[product], stats[product] = tr, st
    if len(grids) > 1:
        t0 = next(iter(grids.values()))
        if any(not t.almost_equals(t0, precision=1e-9) for t in grids.values()):
            issues.append('files not on the same grid')
    rec['issues'] = '; '.join(issues)
    rec['warnings'] = '; '.join(warns)
    return rec, stats

In [ ]:
t0 = time.time()
rows = TableDHS[['cluster_id', 'year', 'lon', 'lat']].to_dict('records')
with multiprocessing.get_context('fork').Pool(N_WORKERS) as pool:
    results = list(tqdm(pool.imap_unordered(check_cluster, rows, chunksize=16), total=len(rows)))
qc = pd.DataFrame([r for r, _ in results])
STATS = {r['cluster_id']: s for r, s in results}
print(f'checked {len(qc)} clusters in {time.time() - t0:.0f} s, {qc["MB"].sum() / 1e3:.1f} GB on disk')

# 3. Decide which clusters are usable

This cell only applies the limits from Settings to the numbers found above, so you can change the limits and re-run it without reading the files again.

In [ ]:
for col in ['landsat_nodata', 'landsat_levels', 'landsat_md5', 'landsat_bright', 'landsat_temp_K',
            'modis_empty', 'modis_empty_bands', 'modis_nodata', 'modis_min', 'modis_max',
            'nl_sensor', 'nl_nodata', 'nl_min', 'nl_max']:
    if col not in qc:                    # a column is absent when no file of that kind could be read
        qc[col] = np.nan
qc['complete'] = (qc[[f'{p}_status' for p in FILES]] == 'ok').all(axis=1)

# identical Landsat patches in different clusters (same place and year, or a copying mistake)
dup = qc['landsat_md5'].notna() & qc['landsat_md5'].duplicated(keep=False)
qc.loc[dup, 'warnings'] = (qc.loc[dup, 'warnings'] + '; landsat: identical to another cluster').str.strip('; ')

labels = TableDHS.set_index('cluster_id')[LABEL]
qc['label_ok'] = qc['cluster_id'].map(labels).between(0, 100).values     # NaN -> False

def excluded_because(r):
    why = []
    if not r['complete']:
        why.append('incomplete or broken')
    if r['issues']:
        why.append('issues')
    for p in FILES:
        if r[f'{p}_nodata'] > MAX_NODATA[p]:
            why.append(f'{p} no-data')
    if r['modis_empty'] > MODIS_MAX_EMPTY_BANDS:
        why.append('empty MODIS bands')
    if not r['label_ok']:
        why.append('label')
    return '; '.join(why)

qc['excluded_because'] = qc.apply(excluded_because, axis=1)
qc['usable'] = qc['excluded_because'] == ''
qc.to_csv(os.path.join(QC_DIR, 'qc_report.csv'), index=False)
print(f"{len(qc)} clusters | complete: {qc['complete'].sum()} | usable for training: {qc['usable'].sum()}")
print('report:', os.path.join(QC_DIR, 'qc_report.csv'))

# 4. What was found

In [ ]:
def count_items(col):
    """How often each '; '-separated item occurs in a report column."""
    s = qc[col].fillna('').str.split('; ').explode()
    return s[s != ''].value_counts()

print('file status\n', qc[[f'{p}_status' for p in FILES]].apply(pd.Series.value_counts).fillna(0).astype(int), '\n')
print('leftover .part files:', int(qc['part_files'].sum()), '| folders not in the CSV:', len(unexpected_folders), '\n')
print('issues (exclude the cluster)\n', count_items('issues').to_string(), '\n')
print('warnings (look at these)\n', count_items('warnings').to_string(), '\n')
print('excluded because\n', count_items('excluded_because').to_string(), '\n')

by_year = qc.groupby('year').agg(clusters=('cluster_id', 'size'), complete=('complete', 'sum'),
                                 usable=('usable', 'sum'), landsat_nodata=('landsat_nodata', 'mean'),
                                 modis_empty=('modis_empty', 'mean'))
print(by_year.round(4).to_string())

# what the download log says about clusters with missing files
if os.path.exists(LOG_CSV):
    last = pd.read_csv(LOG_CSV).drop_duplicates('cluster_id', keep='last').set_index('cluster_id')
    missing = qc.loc[(qc[[f'{p}_status' for p in FILES]] == 'missing').any(axis=1), 'cluster_id']
    print('\ndownload log status of clusters with missing files\n',
          last.reindex(missing)['status'].fillna('not in log (not attempted yet)').value_counts().to_string())

In [ ]:
def hist(ax, values, title, **kw):
    values = pd.Series(values, dtype=float).dropna()
    if len(values):
        ax.hist(values, **kw)
    ax.set_title(title)

fig, ax = plt.subplots(1, 4, figsize=(20, 3.5))
hist(ax[0], qc.loc[qc['landsat_nodata'] > 0, 'landsat_nodata'], 'Landsat no-data share (where > 0)', bins=50)
hist(ax[1], qc['landsat_bright'], 'Landsat share of bright pixels', bins=50)
for sensor, g in qc.groupby('nl_sensor'):
    hist(ax[2], g['nl_max'].clip(lower=0.1), 'NL maximum per patch', bins=np.logspace(-1, 4, 50), alpha=0.6, label=sensor)
ax[2].set_xscale('log')
if qc['nl_sensor'].notna().any():
    ax[2].legend()
empty_counts = qc['modis_empty'].dropna().astype(int).value_counts().sort_index()
ax[3].bar(empty_counts.index.astype(str), empty_counts.values)
ax[3].set_title('empty MODIS bands per file')
plt.show()

In [ ]:
def show(cluster_ids, title=''):
    """Landsat RGB, NL, mean NDVI and the NDVI curve at the cluster, one row per cluster."""
    cluster_ids = list(cluster_ids)
    if not cluster_ids:
        return
    fig, axes = plt.subplots(len(cluster_ids), 4, figsize=(16, 3.4 * len(cluster_ids)), squeeze=False)
    fig.suptitle(title)
    report = qc.set_index('cluster_id')
    for ax, cid in zip(axes, cluster_ids):
        r = report.loc[cid]
        files = {p: os.path.join(cluster_dir(cid), f) for p, f in FILES.items()}
        ax[0].set_title(f'{cid} ({r.year})', fontsize=9)
        if r.landsat_status == 'ok':
            with rasterio.open(files['landsat']) as src:
                ls = src.read(masked=True)
            ax[0].imshow(np.moveaxis(np.clip((ls[[2, 1, 0]].filled(0) * 0.0000275 - 0.2) / 0.3, 0, 1), 0, -1))
        if r.nl_status == 'ok':
            with rasterio.open(files['nl']) as src:
                fig.colorbar(ax[1].imshow(src.read(1, masked=True), cmap='magma'), ax=ax[1])
            ax[1].set_title(f'NL {r.nl_sensor}', fontsize=9)
        if r.modis_status == 'ok':
            with rasterio.open(files['modis']) as src:
                nd = src.read(masked=True)
            fig.colorbar(ax[2].imshow(nd.mean(axis=0), cmap='YlGn', vmin=0, vmax=1), ax=ax[2])
            ax[2].set_title('mean NDVI', fontsize=9)
            ax[3].plot(range(1, 24), nd[:, PATCH_PX // 2, PATCH_PX // 2], 'o-')
        ax[3].set_title((r.issues or r.excluded_because or 'usable')[:70], fontsize=9)
    plt.tight_layout()
    plt.show()

show(qc.loc[qc['usable'], 'cluster_id'].sample(min(6, qc['usable'].sum()), random_state=0), 'random usable clusters')
show(qc.dropna(subset=['landsat_bright']).nlargest(6, 'landsat_bright')['cluster_id'], 'most bright Landsat pixels')
show(qc.loc[~qc['usable'] & (qc['landsat_status'] == 'ok'), 'cluster_id'].head(6), 'excluded clusters')

# 5. Move broken files aside (optional)

Files that are unreadable or fail a structure check, and leftover `.part` files, are moved to `QC_DIR/_quarantine/` with the same folder layout. Running `01` again then downloads them anew. Nothing is moved while `MOVE_BROKEN = False`.

In [ ]:
MOVE_BROKEN = False

to_move = []
for r in qc.itertuples():
    folder = cluster_dir(r.cluster_id)
    to_move += [os.path.join(folder, f) for p, f in FILES.items() if getattr(r, f'{p}_status') in ('broken', 'unreadable')]
    if r.part_files:
        to_move += glob.glob(os.path.join(glob.escape(folder), '*.part'))
print(len(to_move), 'files to move', to_move[:5])

if MOVE_BROKEN:
    for src in to_move:
        dst = os.path.join(QC_DIR, '_quarantine', os.path.relpath(src, OUT_DIR))
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.move(src, dst)
    print('moved to', os.path.join(QC_DIR, '_quarantine'), '- run 01 again to download them')

# 6. Training list and band statistics

In [ ]:
train = TableDHS[TableDHS['cluster_id'].isin(qc.loc[qc['usable'], 'cluster_id'])].copy()
train = train.merge(qc[['cluster_id', 'nl_sensor', 'landsat_levels']], on='cluster_id', how='left')
train['folder'] = [cluster_dir(c) for c in train['cluster_id']]
train.to_csv(os.path.join(QC_DIR, 'train_clusters.csv'), index=False)
print(len(train), 'clusters ->', os.path.join(QC_DIR, 'train_clusters.csv'))
print(train.groupby('year').size().to_string())

# per-band statistics of the usable clusters; NL separately per sensor (DMSP DN 0-63 vs VIIRS radiance)
acc = {}
sensor = dict(zip(qc['cluster_id'], qc['nl_sensor']))
for cid in train['cluster_id']:
    for p, s in STATS[cid].items():
        key = (p, sensor[cid] if p == 'nl' else 'all')
        if key not in acc:
            acc[key] = s.copy()
        else:
            a = acc[key]
            a[:, :3] += s[:, :3]
            a[:, 3] = np.fmin(a[:, 3], s[:, 3])
            a[:, 4] = np.fmax(a[:, 4], s[:, 4])
rows = []
for (p, grp), a in sorted(acc.items()):
    for b, name in enumerate(BANDS[p]):
        n, s1, s2, lo, hi = a[b]
        mean = s1 / n if n else np.nan
        rows.append(dict(product=p, group=grp, band=name, pixels=int(n), mean=mean,
                         std=np.sqrt(max(s2 / n - mean ** 2, 0)) if n else np.nan, min=lo, max=hi))
band_table = pd.DataFrame(rows)
band_table.to_csv(os.path.join(QC_DIR, 'band_stats.csv'), index=False)
print('\n', band_table.round(4).to_string(index=False))

# 7. Load a batch the way training will

Stacks Landsat (7) + NL (1) + MODIS (23) = 31 channels for 64 usable clusters, the order of your earlier TFRecords, and checks that the batch is finite and has no -999999 left. It also shows how long reading one sample takes.

In [ ]:
def read(path):
    with rasterio.open(path) as src:
        return src.read()

sample = train.sample(min(64, len(train)), random_state=1)
if len(sample):
    t0 = time.time()
    batch = np.stack([np.concatenate([read(os.path.join(f, FILES[p])) for p in ('landsat', 'nl', 'modis')])
                      for f in sample['folder']])
    dt = (time.time() - t0) / len(sample)
    print('batch', batch.shape, batch.dtype, f'{batch.nbytes / 1e6:.0f} MB, {dt * 1e3:.0f} ms per sample')
    print('finite:', bool(np.isfinite(batch).all()), '| -999999 values left:', int((batch == NODATA).sum()))
else:
    print('no usable clusters yet')